# 🧠 CalRetail — Conversational Buying Assistant
## Goal
Extract shopper intent, category, brand, color, size and budget from free-text chat, and
suggest matching catalogue items with a natural-language reply.

## Algorithmic Explanation
**LangChain LCEL extraction chain (`ChatPromptTemplate | chat_model | PydanticOutputParser`), with a rule-based safety net**
1. When an LLM provider key is configured, run the shopper's message through an idiomatic
   LangChain chain (`backend.utils.llm_service.llm_structured`) built from a `ChatPromptTemplate`
   and validated into the `BuyingAssistantExtraction` Pydantic schema via `PydanticOutputParser`
   — extracting intent, category, brand, color, size and price range, plus a drafted reply.
2. If no LLM is configured, or the LLM output can't be parsed into that schema, fall back to a
   deterministic regex/keyword parser over the real product catalogue's categories and brands.
3. Filter the live product catalogue on the extracted attributes and return the best-matching,
   lowest-priced items first.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
import re
prod = load_table('products')
cust = load_table('customers')

categories = prod['category'].dropna().unique().tolist()
brands = prod['brand'].dropna().unique().tolist()
print(f"Categories: {len(categories)} | Brands Sample: {brands[:5]}")


In [ ]:
from backend.utils.llm_service import llm_structured
from backend.schemas.llm_schemas import BuyingAssistantExtraction

BUYING_ASSISTANT_SYSTEM_PROMPT = """You are CalRetail's personal shopping assistant for a fashion e-commerce site.
From the shopper's message, extract the exact category, brand, color, size and price range they
are looking for, and draft a short, friendly one-sentence reply.

Valid categories: {categories}
Valid brands (sample): {brands}

Return ONLY a JSON object with this exact shape (use null for anything not mentioned):
{{
  "intent": "buy" | "browse" | "compare" | "budget",
  "category": "<one of the valid categories or null>",
  "brand": "<one of the valid brands or null>",
  "color": "<color mentioned or null>",
  "size": "<size mentioned or null>",
  "min_price": <number or null>,
  "max_price": <number or null>,
  "reply": "<short, friendly one-sentence reply referencing what you understood>"
}}"""


def _extract_intent_rules(message: str, products_df: pd.DataFrame) -> dict:
    """Regex/keyword-based intent & entity extraction — the safety net used when no LLM
    provider is configured, or if the LLM response can't be parsed as JSON."""
    msg_lower = message.lower()
    intent = "browse"
    if any(w in msg_lower for w in ["buy", "order", "purchase"]): intent = "buy"
    elif any(w in msg_lower for w in ["compare", "vs"]): intent = "compare"
    elif any(w in msg_lower for w in ["budget", "cheap", "under"]): intent = "budget"

    cats = products_df['category'].dropna().unique().tolist()
    brds = products_df['brand'].dropna().unique().tolist()
    matched_cats = [c for c in cats if c.lower() in msg_lower]
    det_cat = max(matched_cats, key=len) if matched_cats else None
    matched_brands = [b for b in brds if b.lower() in msg_lower]
    det_brand = max(matched_brands, key=len) if matched_brands else None

    max_price, min_price = None, None
    range_match = re.search(r"(?:rs\.?|₹)?\s*(\d+)\s*(?:-|to)\s*(?:rs\.?|₹)?\s*(\d+)", msg_lower)
    if range_match:
        min_price, max_price = int(range_match.group(1)), int(range_match.group(2))
    else:
        price_match = re.search(r"under\s*(?:rs\.?|₹)?\s*(\d+)", msg_lower)
        if price_match: max_price = int(price_match.group(1))

    return {
        "intent": intent, "category": det_cat, "brand": det_brand,
        "color": None, "size": None, "min_price": min_price, "max_price": max_price,
        "reply": None,
    }


def _llm_extract_intent(message: str):
    """LangChain-backed extraction via an idiomatic `ChatPromptTemplate | llm |
    PydanticOutputParser` chain. Returns None if the LLM output can't be parsed/validated,
    so the caller can transparently fall back to the rule-based parser."""
    prompt = BUYING_ASSISTANT_SYSTEM_PROMPT.format(
        categories=", ".join(categories), brands=", ".join(brands[:20])
    )
    result = llm_structured(prompt, message, BuyingAssistantExtraction)
    return result.model_dump() if result is not None else None


def process_chat_message(cust_id, message):
    extracted = _llm_extract_intent(message)
    used_llm = extracted is not None
    if extracted is None:
        extracted = _extract_intent_rules(message, prod)

    intent     = extracted.get("intent") or "browse"
    det_cat    = extracted.get("category")
    det_brand  = extracted.get("brand")
    color      = extracted.get("color")
    size       = extracted.get("size")
    max_price  = extracted.get("max_price")
    min_price  = extracted.get("min_price")

    filtered = prod[prod['is_active'] == True] if 'is_active' in prod.columns else prod
    if det_cat and det_cat in categories: filtered = filtered[filtered['category'] == det_cat]
    if det_brand and det_brand in brands: filtered = filtered[filtered['brand'] == det_brand]
    if color and 'color' in filtered.columns:
        filtered = filtered[filtered['color'].astype(str).str.lower() == str(color).lower()]
    if size and 'size' in filtered.columns:
        filtered = filtered[filtered['size'].astype(str).str.lower() == str(size).lower()]
    if max_price: filtered = filtered[filtered['price'] <= max_price]
    if min_price: filtered = filtered[filtered['price'] >= min_price]

    suggestions = filtered.sort_values('price').head(5) if not filtered.empty else filtered.head(0)

    cname = cust[cust['customer_id'] == cust_id].iloc[0]['name'].split()[0] if cust_id in cust['customer_id'].values else "Shopper"
    reply = extracted.get("reply")
    if not reply:
        reply = f"Hi {cname}! Based on your search, here are some great {det_cat or 'selections'} for you."

    return {
        "intent": intent, "category": det_cat, "brand": det_brand, "price_limit": max_price,
        "response": reply,
        "powered_by": "LangChain LLM" if used_llm else "Rule-Based Engine",
        "suggestions": suggestions[['product_id', 'product_name', 'category', 'brand', 'price']].to_dict(orient='records')
    }

In [ ]:
query = "Show me some shirts under Rs. 1500"
backend_res = process_chat_message("C00001", query)
print("=== CALRETAIL ASSISTANT CONVERSATION ===")
print(f"User: '{query}'")
print(f"Assistant: {backend_res['response']}")
print("Matched Items:")
for sug in backend_res['suggestions']:
    print(f"  - {sug['product_name']} | Brand: {sug['brand']} | Price: ₹{sug['price']}")
